## Подготовка

In [1]:
# Автоперезагрузка модулей из src при их изменении
%load_ext autoreload
%autoreload 2

In [3]:
# Базовые библиотеки
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from omegaconf import OmegaConf

# Функции из src
from src.data import load_data, save_submission, build_preprocessor
from src.validation import cross_validate, log_result
from src.models import build_model
from src.features import add_features

In [4]:
# Загрузка конфига и данных
config = OmegaConf.load("config.yaml")
X, y, X_test, test_ids = load_data(config)

In [5]:
# Хелпер: собрать модель из конфига, прогнать CV и записать результат в журнал
def run_model(cfg):
    model = build_model(cfg)
    scores, _ = cross_validate(model, X, y, X_test, cfg)
    features = list(cfg.features.numeric) + list(cfg.features.categorical) + list(cfg.features.log_numeric)
    params = OmegaConf.to_container(cfg.model.params, resolve=True)
    log_result(cfg, cfg.model.name, params, features, scores)
    return round(scores.mean(), 4), round(scores.std(), 4)

In [6]:
X.shape, X_test.shape, y.head(3)

((1460, 79),
 (1459, 79),
 0    12.247699
 1    12.109016
 2    12.317171
 Name: SalePrice, dtype: float64)

## Бейзлайн

In [7]:
# 3.2 Бейзлайн: всем домам медианная цена
from sklearn.dummy import DummyRegressor

scores, _ = cross_validate(DummyRegressor(strategy="median"), X, y, X_test, config)
log_result(config, "median", {}, [], scores)
round(scores.mean(), 4), round(scores.std(), 4)

(np.float64(0.3998), np.float64(0.0118))

In [8]:
# 3.3 Линейная регрессия на 4 сильнейших признаках: журнал и сабмит
print(run_model(config))
_, test_pred = cross_validate(build_model(config), X, y, X_test, config)
save_submission(test_ids, test_pred, config, "baseline_linreg_4features")

(np.float64(0.1842), np.float64(0.0296))


'outputs\\submission_baseline_linreg_4features.csv'